# 🔬 Generate verbatims with Qwen2.5-7B-Instruct on Colab GPU

This notebook generates **7,043 customer verbatims** using a local HuggingFace
model. Designed to run on Google Colab with a T4 (free) or L4 (Pro) GPU.

**Workflow**:
1. Mount Google Drive (or upload directly).
2. Upload `verbatim_prompts.parquet` from your local project.
3. Load Qwen2.5-7B-Instruct in 4-bit (fits in T4's 16 GB).
4. Generate verbatims in batches (~30 min on L4, ~1h on T4).
5. Download `verbatims.parquet` back to your local machine.

**Make sure you switch the runtime to GPU**: Runtime → Change runtime type → T4 / L4 GPU.

## 1. Install dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas pyarrow tqdm

## 2. Mount Google Drive (optional but recommended)

If you'd rather upload the prompts file directly, skip this cell and use the
"Files" tab in Colab to upload `verbatim_prompts.parquet`.

In [ ]:
# Optional — mount Drive
from google.colab import drive
drive.mount('/content/drive')

## 3. Load the prompts file

Adjust `PROMPTS_PATH` below to wherever you uploaded the file.

In [ ]:
PROMPTS_PATH = "/content/verbatim_prompts.parquet"  # or "/content/drive/MyDrive/..." if Drive
OUTPUT_PATH = "/content/verbatims.parquet"

import pandas as pd

prompts = pd.read_parquet(PROMPTS_PATH)
print(f"Loaded {len(prompts):,} prompts")
print(f"Columns: {prompts.columns.tolist()}")
print(f"\nExpected class distribution:")
print(prompts["expected_class"].value_counts())
print(f"\nCounter-intuitive rate: {prompts['counter_intuitive'].mean():.1%}")
prompts.head(3)

## 4. Load Qwen2.5-7B-Instruct

We load in **4-bit quantization** to fit in T4's 16 GB. Quality is barely
affected for short generation tasks like this.

**~5 minutes on first run** (model download + load).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"✓ Model loaded on {model.device}")
print(f"  Tokenizer vocab size: {len(tokenizer):,}")

## 5. Generation function

We use Qwen's chat template + greedy-ish sampling (low temperature) for stability,
with `do_sample=True` to inject lexical variety so the verbatims aren't all clones.

In [ ]:
def format_chat(system_prompt: str, user_prompt: str) -> str:
    """Apply Qwen2.5 chat template."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

@torch.inference_mode()
def generate_batch(system_prompts, user_prompts, max_new_tokens=120):
    """Generate a batch of verbatims."""
    formatted = [format_chat(s, u) for s, u in zip(system_prompts, user_prompts)]
    inputs = tokenizer(
        formatted, return_tensors="pt", padding=True, truncation=True,
        max_length=512,
    ).to(model.device)
    
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    # Slice off the prompt tokens
    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [t.strip() for t in decoded]

# Quick smoke test
sample = prompts.iloc[:4]
samples_out = generate_batch(
    sample["system_prompt"].tolist(),
    sample["user_prompt"].tolist(),
    max_new_tokens=120,
)
for i, (idx, row) in enumerate(sample.iterrows()):
    print(f"\n[{idx}] expected={row['expected_class']}, ci={row['counter_intuitive']}")
    print(f"  → {samples_out[i]}")

## 6. Generate all 7,043 verbatims

Runs in batches of 8. Reproducibility: we set the torch seed before generation.

In [ ]:
import torch
from tqdm.auto import tqdm
import time

# Reproducibility — anyone running this notebook with the same prompts file gets the same output
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

BATCH_SIZE = 8

verbatims = []
n_total = len(prompts)
start = time.time()

for i in tqdm(range(0, n_total, BATCH_SIZE), desc="Generating"):
    batch = prompts.iloc[i:i + BATCH_SIZE]
    out = generate_batch(
        batch["system_prompt"].tolist(),
        batch["user_prompt"].tolist(),
    )
    verbatims.extend(out)

elapsed = time.time() - start
print(f"\n✓ Generated {len(verbatims):,} verbatims in {elapsed / 60:.1f} min")
print(f"  Rate: {len(verbatims) / elapsed:.1f} verbatims/sec")

## 7. Quality check

In [ ]:
import pandas as pd
import numpy as np

result = prompts.copy()
result["verbatim"] = verbatims

# Length stats
lens = result["verbatim"].str.len()
print("Verbatim length (chars):")
print(f"  mean   = {lens.mean():.0f}")
print(f"  median = {lens.median():.0f}")
print(f"  min    = {lens.min()}")
print(f"  max    = {lens.max()}")
print(f"  below 30 chars (suspicious): {(lens < 30).sum()}")

# Forbidden words check (per the system prompt)
FORBIDDEN = ["detractor", "passive", "promoter", "nps", "satisfaction score"]
text_lower = result["verbatim"].str.lower()
print(f"\nForbidden words check:")
for w in FORBIDDEN:
    n = text_lower.str.contains(w, regex=False).sum()
    flag = "⚠" if n > 0 else "✓"
    print(f"  {flag} '{w}': {n} hits")

# Duplicates
n_dup = result["verbatim"].duplicated().sum()
print(f"\nDuplicate verbatims: {n_dup} (model collapse if many)")

# Per-class samples
print("\n" + "=" * 60)
print("SAMPLE — 3 per class")
print("=" * 60)
for cls in ["Detractor", "Passive", "Promoter"]:
    print(f"\n--- {cls} ---")
    sub = result[result["expected_class"] == cls].sample(3, random_state=42)
    for _, row in sub.iterrows():
        ci = " [CI]" if row.get("counter_intuitive", False) else ""
        print(f"\n[{row.name}]{ci}: \"{row['verbatim']}\"")

## 8. Save and download

Save the verbatims and download the file. Place it in your local project's
`data/external/` folder, then run `make load-verbatims`.

In [ ]:
# Keep only what's needed downstream — drop the heavy prompt columns
output = result[["verbatim", "expected_class", "counter_intuitive"]].copy()
output.to_parquet(OUTPUT_PATH)
print(f"✓ Saved {len(output):,} verbatims to {OUTPUT_PATH}")
print(f"  File size: {pd.io.common.file_path_to_url(OUTPUT_PATH)}")

# Download via Colab
from google.colab import files
files.download(OUTPUT_PATH)
print("\n→ Place the downloaded file in your local data/external/ folder.")

## 9. Done

Next steps on your local machine:

```bash
# Move the downloaded file
mv ~/Downloads/verbatims.parquet data/external/verbatims.parquet

# Integrate into the dataset
make load-verbatims
make inspect-verbatims    # quality audit
make test-phase5          # invariants

# Optional — local notebook for figures
jupyter notebook notebooks/05_verbatims_inspection.ipynb
```